# Snow4Flow Glacier Grids

*Andy Aschwanden, April 2026*

Here are some thoughts on S4F Glacier Grids:

  - Grids, CRS and associated properties should be chosen by the downstream users (the modeling community) by data collectors. This reduces the amount of reprojections needed.
  - point data can be in EPSG:4326
  - Ellipsoid vs Geoid: For modeling purposes we need sea level to be at 0m. Removing or adding geoids/ellipsoids are a pain and great care is needed to avoid ice thicknesses < 0.

In [ ]:
%pip install xarray h5netcdf rioxarray geopandas

In [ ]:
from pathlib import Path
from urllib.parse import urlparse

import boto3
from boto3.s3.transfer import TransferConfig
from botocore.config import Config
from tqdm import tqdm

import geopandas as gpd
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pylab as plt
import matplotlib as mpl
from matplotlib.patches import FancyArrow
from shapely.geometry import Point, box

import cartopy.crs as ccrs 

In [ ]:
proj = ccrs.PlateCarree()

fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

In [ ]:
def north_arrow(ax, x=0.95, y=0.15, arrow_length=0.1,                                                                                                                                   
                label="N", fontsize=14, lw=2, color="black"):
    """
    Draw a true-north arrow at axes-fraction (x, y).                                                                                                                                 
    Works for any Cartopy projection. The arrow points along the local                                                                                                                  
    meridian at the anchor's geographic location.                                                                                                                                       
    """
    proj = ax.projection                                                                                                                                                                
    pc = ccrs.PlateCarree()                                                                                                                                                             
   
    # 1. anchor (axes fraction) -> display -> data (projection) -> lon/lat                                                                                                              
    disp = ax.transAxes.transform((x, y))
    px, py = ax.transData.inverted().transform(disp)                                                                                                                                    
    lon, lat = pc.transform_point(px, py, proj)
    
    # 2. a point a small step north in lon/lat, back into projection coords                                                                                                             
    dlat = 0.5  # degrees; small enough to be "local"
    px2, py2 = proj.transform_point(lon, lat + dlat, pc)                                                                                                                                
                                                                                                                                                                                          
    # 3. direction in *display* coords so length is visually consistent                                                                                                                 
    d0 = ax.transData.transform((px, py))                                                                                                                                               
    d1 = ax.transData.transform((px2, py2))                                                                                                                                             
    dx, dy = d1 - d0
    angle = np.arctan2(dy, dx)                                                                                                                                                          
                                                                                                                                                                                          
    # 4. draw arrow + label in axes-fraction space, rotated to that angle                                                                                                               
    ux, uy = np.cos(angle), np.sin(angle)                                                                                                                                               
    # convert arrow_length (axes fraction) to a vector in axes coords                                                                                                                   
    head = (x + ux * arrow_length, y + uy * arrow_length)                                                                                                                               
    tail = (x - ux * arrow_length * 0.2, y - uy * arrow_length * 0.2)                                                                                                                   
                                                                                                                                                                                          
    ax.annotate(                                                                                                                                                                        
        "", xy=head, xytext=tail,                                                                                                                                                       
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="-|>", lw=lw, color=color,                                                                                                                           
                        mutation_scale=20),                                                                                                                                             
    )                                                                                                                                                                                   
    ax.text(                                                                                                                                                                            
          head[0] + ux * 0.02, head[1] + uy * 0.02, label,
          transform=ax.transAxes,                                                                                                                                                         
          ha="center", va="center",
          fontsize=fontsize, fontweight="bold", color=color,                                                                                                                              
      )           

def download_from_s3(s3_uri: str, dest: str | Path) -> Path:
    """
    Download a file from AWS S3.

    Parameters
    ----------
    s3_uri : str
        URI of S3 object to download.
    dest : str or Path
        Path to the downloaded file.

    Returns
    -------
    Path
        Path to the downloaded file.
    """
    dest = Path(dest)

    parsed_url = urlparse(s3_uri)
    bucket = parsed_url.netloc
    prefix = parsed_url.path.lstrip("/")

    s3 = boto3.client("s3")
    head = s3.head_object(Bucket=bucket, Key=prefix)
    total_size = head["ContentLength"]

    with tqdm(total=total_size, unit="B", unit_scale=True, desc=dest.name) as pbar:
        s3.download_file(bucket, prefix, str(dest), Callback=pbar.update)

    return dest

def get_glacier_from_rgi_id(rgi: gpd.GeoDataFrame | str | Path, rgi_id: str) -> gpd.GeoDataFrame:
    """
    Return the row in the GeoDataFrame matching the given RGI ID.

    Parameters
    ----------
    rgi : geopandas.GeoDataFrame
        GeoDataFrame containing glacier data.
    rgi_id : str
        RGI identifier to look up.

    Returns
    -------
    geopandas.GeoSeries
        The matching row.
    """
    if isinstance(rgi, str | Path):
        rgi = gpd.read_file(rgi)

    glacier = rgi[rgi["rgi_id"] == rgi_id]
    return glacier

def create_domain(
    x_bnds: list | np.ndarray,
    y_bnds: list | np.ndarray,
    resolution: float | None = None,
    x_dim: str = "x",
    y_dim: str = "y",
    crs: str = "EPSG:3413",
) -> xr.Dataset:
    """
    Create an xarray.Dataset representing a domain with specified x and y boundaries.

    Parameters
    ----------
    x_bnds : list or numpy.ndarray
        A list or array containing the minimum and maximum x-coordinate boundaries.
    y_bnds : list or numpy.ndarray
        A list or array containing the minimum and maximum y-coordinate boundaries.
    resolution : float or None, optional
        The resolution of the grid, by default None.
    x_dim : str, optional
        The name of the x dimension, by default "x".
    y_dim : str, optional
        The name of the y dimension, by default "y".
    crs : str, optional
        The coordinate reference system (CRS) for the domain, by default "EPSG:3413".

    Returns
    -------
    xarray.Dataset
        An xarray.Dataset containing the domain information, including coordinates,
        boundary data, and mapping attributes.

    Notes
    -----
    The dataset includes:
    - `x` and `y` coordinates with associated metadata.
    - A `mapping` DataArray with polar stereographic projection attributes.
    - A `domain` DataArray with a reference to the `mapping`.
    - `x_bnds` and `y_bnds` DataArrays representing the boundaries of the domain.

    Examples
    --------
    >>> x_bnds = [0, 1000]
    >>> y_bnds = [0, 2000]
    >>> ds = create_domain(x_bnds, y_bnds)
    >>> print(ds)
    """

    if resolution is not None:
        # np.arange(start, stop) includes start but not stop
        xb = np.arange(x_bnds[0], x_bnds[1] + resolution, resolution)
        yb = np.arange(y_bnds[0], y_bnds[1] + resolution, resolution)
        x = np.arange(x_bnds[0] + resolution / 2, x_bnds[1] + resolution - resolution / 2, resolution)
        y = np.arange(y_bnds[0] + resolution / 2, y_bnds[1] + resolution - resolution / 2, resolution)
        x_bounds = np.stack([xb[:-1], xb[1:]]).T
        y_bounds = np.stack([yb[:-1], yb[1:]]).T
    else:
        x = np.array([0])
        y = np.array([0])
        x_bounds = np.array([[x_bnds[0], x_bnds[1]]])
        y_bounds = np.array([[y_bnds[0], y_bnds[1]]])

    x_bnds_dim = f"{x_dim}_bnds"
    y_bnds_dim = f"{y_dim}_bnds"
    coords = {
        x_dim: (
            [x_dim],
            x,
            {
                "axis": x_dim.upper(),
                "bounds": x_bnds_dim,
            },
        ),
        y_dim: (
            [y_dim],
            y,
            {
                "axis": y_dim.upper(),
                "bounds": y_bnds_dim,
            },
        ),
    }
    ds = xr.Dataset(
        {
            "domain": xr.DataArray(
                data=0,
                dims=[y_dim, x_dim],
                coords={x_dim: coords[x_dim], y_dim: coords[y_dim]},
                attrs={
                    "dimensions": f"{x_dim} {y_dim}",
                },
            ),
            x_bnds_dim: xr.DataArray(
                data=x_bounds,
                dims=[x_dim, "nv2"],
                coords={x_dim: coords[x_dim]},
            ),
            y_bnds_dim: xr.DataArray(
                data=y_bounds,
                dims=[y_dim, "nv2"],
                coords={y_dim: coords[y_dim]},
            ),
        },
        attrs={"Conventions": "CF-1.8"},
    ).rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim)
    ds.rio.write_crs(crs, inplace=True).rio.write_coordinate_system(inplace=True)
    for var in list(ds.data_vars) + list(ds.coords):
        ds[var].encoding["_FillValue"] = None

    return ds

def grid_points_from_dataset(ds: xr.Dataset):
    # Cell centers as points
    points = [Point(xi, yi) for yi in ds.y.values for xi in ds.x.values]                                                                                                                       
    gdf_points = gpd.GeoDataFrame(geometry=points, crs=ds.spatial_ref.attrs.get("crs_wkt"))                                                                                                    

    return gdf_points
    
def grid_cells_from_dataset(ds: xr.Dataset):
    # Cell faces as polygons from bounds                                                                                                                                                       
    polys = []                                                                                                                                                                                 
    for j in range(len(ds.y)):                                                                                                                                                                 
        for i in range(len(ds.x)):
            polys.append(box(                                                                                                                                                                  
                ds.x_bnds.values[i, 0], ds.y_bnds.values[j, 0],                                                                                                                                
                ds.x_bnds.values[i, 1], ds.y_bnds.values[j, 1],
            ))                                                                                                                                                                                 
    gdf_cells = gpd.GeoDataFrame(                             
        {"domain": ds.domain.values.flatten()},                                                                                                                                                
        geometry=polys,                                                                                                                                                                        
        crs=ds.spatial_ref.attrs.get("crs_wkt"),            
    )   
    return gdf_cells

## Download RGI v7

This is the same as on NSIDC but in GPKG format

For all modeling purposes we will be using the glacier complexes `C` of RGI not the indiviual glaciers `G`. The reason is related to how lateral boundary conditions are applied, using complexes avoids the complexity of lateral boundaries with ice thickness

In [ ]:
bucket = "pism-cloud-data"
prefix = "glaciers/rgi"

print("RGI Glacier Complexes (C)")
rgi_c_file = "rgi_c_01.gpkg"
rgi_c_s3_uri = f"""s3://{bucket}/{prefix}/{rgi_c_file}"""
rgi_c_local = Path(".") / rgi_c_file

if not rgi_c_local.exists():
    print(f"Downloading {rgi_c_s3_uri} -> {rgi_c_local}")
    download_from_s3(rgi_c_s3_uri, rgi_c_local)
else:
    print(f"Using cached {rgi_c_local}")
rgi_c = gpd.read_file(rgi_c_local)

rgi_c_file = "rgi_c_01.gpkg"
bucket = "pism-cloud-data"
prefix = "glaciers/rgi"

print("RGI Glaciers (G)")
rgi_g_file = "rgi_g_01.gpkg"
rgi_g_s3_uri = f"""s3://{bucket}/{prefix}/{rgi_g_file}"""
rgi_g_local = Path(".") / rgi_g_file

if not rgi_g_local.exists():
    print(f"Downloading {rgi_g_s3_uri} -> {rgi_g_local}")
    download_from_s3(rgi_g_s3_uri, rgi_g_local)
else:
    print(f"Using cached {rgi_g_local}")
rgi_g = gpd.read_file(rgi_g_local)

## Select a glacier

Wrangell Glacier Complex

rgi_id = "RGI2000-v7.0-C-01-04374"

In [ ]:
rgi_id = "RGI2000-v7.0-C-01-04374"

## Option 1: EPSG:3413

In [ ]:
# Get glacier and plot in dst_crs
dst_crs = "EPSG:3413"
dst_proj = ccrs.NorthPolarStereo(central_longitude=-45, true_scale_latitude=70) 
glacier = get_glacier_from_rgi_id(rgi_c, rgi_id)
glacier_projected = glacier.to_crs(dst_crs)

In [ ]:
buffer_dist = 5000.0
bounds = glacier_projected.buffer(buffer_dist).geometry.bounds

### Grid Bounds and Resolution

For ease of visualization, we set `dx=5000`, however the final grids are expected to be around `dx ~ 500m`. 

In [ ]:
# Round in projected meters to a dx=5000 m grid
dx = 5000
x_min = np.ceil((bounds.minx.item()) / dx) * dx
x_max = np.floor((bounds.maxx.item()) / dx) * dx
y_min = np.ceil((bounds.miny.item()) / dx) * dx
y_max = np.floor((bounds.maxy.item()) / dx) * dx

In [ ]:
target_grid = create_domain([x_min, x_max], [y_min, y_max], resolution=dx, crs=dst_crs)
gdf_cells = grid_cells_from_dataset(target_grid)
gdf_points = grid_points_from_dataset(target_grid)

### The Glacier Grid in EPSG:3413

Compared to `Alaska Albers` or `UTM`, Wrangell is rotated by 90 degrees.

In [ ]:
with mpl.rc_context(rc=rc_params): 
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": dst_proj})
    gdf_cells.boundary.plot(ax=ax, color="gray", lw=0.3, transform=dst_proj)                                                                                                                                       
    gdf_points.plot(ax=ax, color="red", markersize=0.5, transform=dst_proj)
    glacier_projected.boundary.plot(ax=ax, transform=dst_proj)
    north_arrow(ax, x=0.92, y=0.15, arrow_length=0.08)


## Option B: Local UTM Zone from RGI7

In [ ]:
# Get glacier and plot in dst_crs
glacier = get_glacier_from_rgi_id(rgi_c, rgi_id)
glacier_series = glacier.iloc[0]
dst_crs = glacier_series["epsg"]
dst_proj = ccrs.epsg(glacier_series["epsg_code"])
glacier_projected = glacier.to_crs(dst_crs)

In [ ]:
buffer_dist = 5000.0
bounds = glacier_projected.buffer(buffer_dist).geometry.bounds

In [ ]:
# Round in projected meters to a dx=1000 m grid
dx = 5000
x_min = np.ceil((bounds.minx.item()) / dx) * dx
x_max = np.floor((bounds.maxx.item()) / dx) * dx
y_min = np.ceil((bounds.miny.item()) / dx) * dx
y_max = np.floor((bounds.maxy.item()) / dx) * dx

In [ ]:
target_grid = create_domain([x_min, x_max], [y_min, y_max], resolution=dx, crs=dst_crs)
gdf_cells = grid_cells_from_dataset(target_grid)
gdf_points = grid_points_from_dataset(target_grid)

### The Glacier Grid in the local UTM

In [ ]:
with mpl.rc_context(rc=rc_params): 
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": dst_proj})
    gdf_cells.boundary.plot(ax=ax, color="gray", lw=0.3)                                                                                                                                       
    gdf_points.plot(ax=ax, color="red", markersize=0.5)
    glacier_projected.boundary.plot(ax=ax)
    north_arrow(ax, x=0.92, y=0.15, arrow_length=0.08)

## Option C: Alaska Albers EPSG:3338

In [ ]:
# Get glacier and plot in dst_crs
dst_crs = "EPSG:3338"
dst_proj = ccrs.epsg(3338)
glacier = get_glacier_from_rgi_id(rgi_c, rgi_id)
glacier_projected = glacier.to_crs(dst_crs)

In [ ]:
buffer_dist = 5000.0
bounds = glacier_projected.buffer(buffer_dist).geometry.bounds

In [ ]:
# Round in projected meters to a dx=1000 m grid
dx = 5000
x_min = np.ceil((bounds.minx.item()) / dx) * dx
x_max = np.floor((bounds.maxx.item()) / dx) * dx
y_min = np.ceil((bounds.miny.item()) / dx) * dx
y_max = np.floor((bounds.maxy.item()) / dx) * dx

In [ ]:
target_grid = create_domain([x_min, x_max], [y_min, y_max], resolution=dx, crs=dst_crs)
gdf_cells = grid_cells_from_dataset(target_grid)
gdf_points = grid_points_from_dataset(target_grid)

### The Glacier Grid in Alaska Albers

In [ ]:
with mpl.rc_context(rc=rc_params): 
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": dst_proj})
    gdf_cells.boundary.plot(ax=ax, color="gray", lw=0.3)                                                                                                                                       
    gdf_points.plot(ax=ax, color="red", markersize=0.5)
    glacier_projected.boundary.plot(ax=ax)
    north_arrow(ax, x=0.92, y=0.15, arrow_length=0.08)

## GlacierMIP 4

The ice thickness dataset proposed for GlacierMIP4 (https://arxiv.org/abs/2512.11685) provides 1 GeoTiff per glacier in the UTM Zone defined by RGI `G`. For large complexes like the Bering-Bagely-Malaspina complex, glaciers of the same complex can have different UTM zone.

In [ ]:
rgi_id = "RGI2000-v7.0-C-01-09429"

bering_complex = rgi_g[rgi_g["rgi_id_c"] == rgi_id]
with mpl.rc_context(rc=rc_params):    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6), subplot_kw={"projection": proj})
    bering_complex.plot(column="utm_zone", ax=ax, alpha=0.5)
    ax.coastlines(linewidth=0.5)

## ITS_LIVE

ITS_LIVE uses local UTM zones plus the NSIDC North and South Polar Stereographic projection. The plot shows that ITS_LIVE uses the same (`EPSG:3413`) projection for all of Alaska.

In [ ]:
itslive_catalog = gpd.read_file("https://its-live-data.s3.amazonaws.com/datacubes/catalog_v02.json")
itslive_catalog.epsg.unique()

In [ ]:
proj = ccrs.PlateCarree()

fontsize = 6
rc_params = {
    "axes.linewidth": 0.15,
    "xtick.major.size": 2.0,
    "xtick.major.width": 0.15,
    "ytick.major.size": 2.0,
    "ytick.major.width": 0.15,
    "hatch.linewidth": 0.15,
    "font.size": fontsize,
    "font.family": "DejaVu Sans",
}

with mpl.rc_context(rc=rc_params):    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6), subplot_kw={"projection": proj})
    itslive_catalog.plot(column="epsg", ax=ax, cmap="jet", alpha=0.5)
    ax.set_xlim(rgi_c.total_bounds[0], rgi_c.total_bounds[2])
    ax.set_ylim(rgi_c.total_bounds[1], rgi_c.total_bounds[3])

    ax.coastlines(linewidth=0.5)


## Discussion

## Recommendations

  - To reproject from a nominal grid cell center in 3413, you transport that center coordinate as $x_p = x_0 + \epsilon * v$ and $x_m = x_0 - \epsilon * v$.  Then you transform $y_p = P(x_p)$ and $y_m = P(x_m)$ to the new CRS $P$, and compute $v_P = (y_p - y_m)/(2 \epsilon)$.  This is just a finite difference approximation to the jacobian-vector product for coordinate transformation - trivial to implement and the error is negligible.
  - While Alaska Albers is the obvious choice for Alaska, what projection should be used for other regions? Using UTM has the advantage that it is predictable for every region you know in advance what projection is used.